#  Linear Algebra with Python
### An Interactive Introduction for  Students

---

Welcome! This notebook will show you how Python can do the heavy lifting for the linear algebra topics you've been studying. Don't worry if you've never coded before.

Every step is explained, and you'll mostly just be **modifying existing code** rather than writing it from scratch.

The notebook has a few custom written functions.  You are not responsible for understanding that code.  Just run the cell and then you will be able to call the function and have it work for you in subsequent cells.

**How to use this notebook:**
- Each gray box (called a **cell**) contains either explanation text or Python code.
- To **run a code cell**, click on it and press **Shift + Enter** (or click the ▶ button in the toolbar).
- Run cells **in order from top to bottom** -- this is important! Later cells use
  functions and variables defined in earlier cells, so if you skip a cell or run
  them out of order you may get an error. If something is not working, try going
  back to the top and running every cell in order again.
- Don't be afraid to experiment! You can always re-run a cell after changing it.

**Estimated time:** around 60 minutes

The notebook covers the following topics in order:

- Vectors and matrix equations
- Solving linear systems and row reduction
- Detecting consistency (does a system have a solution)?
- Linear independence
- One-to-one and onto
- The matrix inverse
- LU factorization

For each topic there is a worked example to read and run. Work through the notebook in order
and make sure each cell runs successfully before moving on.  At the end there are five
problems where you modify the code yourself, but feel free to look back at the examples.

---

##  Step 0: Setting Up — Run This First!

Before anything else, we need to import **NumPy** and **SciPy** which are two powerful Python libraries for mathematics. Think of this as opening your math toolbox.

In [ ]:
# Lines that start with # are "comments" — Python ignores them.
# They're just notes for humans reading the code!

import numpy as np           # NumPy: our main tool for matrices and vectors
from scipy import linalg     # SciPy: gives us LU factorization and more

# This just sets how many decimal places are displayed
np.set_printoptions(precision=4, suppress=True)

print("Libraries loaded successfully! You're ready to go.")

---

## Part 1: Vectors and Matrix Equations

### 1.1 Creating Vectors and Matrices

In Python (using NumPy), we create vectors and matrices using `np.array(...)`.  
- A **vector** is a list of numbers: `[1, 2, 3]`  
- A **matrix** is a list of rows, where each row is itself a list: `[[1, 2], [3, 4]]`

Let's create some and explore them.

In [ ]:
# --- Creating vectors ---
u = np.array([1, 3, -2])   # A vector in R^3
v = np.array([4, 0, 5])    # Another vector in R^3

print("Vector u =", u)
print("Vector v =", v)

# --- Vector addition ---
print("\nu + v =", u + v)

# --- Scalar multiplication ---
print("3 * u =", 3 * u)



In [ ]:
# --- Creating a matrix ---
# Each inner list [ ] is one ROW of the matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("Matrix A:")
print(A)

# shape tells us the size: (rows, columns)
print("\nShape of A:", A.shape)

### 1.2 Matrix-Vector Multiplication: $A\mathbf{x} = \mathbf{b}$

Recall that the matrix equation $A\mathbf{x} = \mathbf{b}$ is equivalent to a system of linear equations.  
In Python, we use `@` for matrix multiplication .

In [ ]:
# The matrix A represents the coefficients of a system
A = np.array([[2, 1, -1],
              [1, 3,  2],
              [1, 0,  1]])

# A candidate solution vector x
x = np.array([1, 2, -1])

# Compute Ax
b = A @ x     # The @ symbol means matrix multiplication

print("A =\n", A)
print("\nx =", x)
print("\nA @ x = b =", b)
print("\nThis means x = [1, 2, -1] IS a solution to the system Ax = b where b =", b)

---

## Part 2: Solving Linear Systems and Row Reduction

### 2.1 Solving $A\mathbf{x} = \mathbf{b}$ Directly

You've practiced row reduction by hand — tedious for large systems! Python can solve $A\mathbf{x} = \mathbf{b}$ instantly using `np.linalg.solve(A, b)`.  

**Important:** This only works when $A$ is square *and* the system has a unique solution.

In [ ]:
# System of 3 equations, 3 unknowns:
#   2*x1 +  x2 -  x3 =  8
#  -3*x1 -  x2 + 2*x3 = -11
#  -2*x1 +  x2 + 2*x3 =  -3

A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)

b = np.array([8, -11, -3], dtype=float)

# Solve the system!
x = np.linalg.solve(A, b)

print("Solution vector x:")
print(f"  x1 = {x[0]:.4f}")
print(f"  x2 = {x[1]:.4f}")
print(f"  x3 = {x[2]:.4f}")

# Verify: compute Ax and check it equals b
print("\nVerification — A @ x should equal b:")
print("  A @ x =", A @ x)
print("  b     =", b)
print("  Match?:", np.allclose(A @ x, b))

### 2.2 Row Echelon Form and the Augmented Matrix

NumPy doesn't have a built-in row-reduction function, but we can write one ourselves — and also use it to look at the augmented matrix $[A | \mathbf{b}]$.  

The function below performs **row reduction** to get row echelon form. You don't need to understand this snippet, just run the cell below and we will be able to use the function.

In [ ]:
# ---------------------------------------------------------------
# This function takes a matrix M and returns its row echelon form.
# You do NOT need to modify this — just run it to define the function.
# ---------------------------------------------------------------
def row_echelon(M):
    """Returns the row echelon form of matrix M (does not modify M)."""
    M = M.astype(float).copy()  # work on a copy, use floats
    rows, cols = M.shape
    pivot_row = 0
    for col in range(cols):
        # Find a non-zero entry in this column (at or below pivot_row)
        nonzero = None
        for r in range(pivot_row, rows):
            if abs(M[r, col]) > 1e-10:
                nonzero = r
                break
        if nonzero is None:
            continue
        # Swap rows
        M[[pivot_row, nonzero]] = M[[nonzero, pivot_row]]
        # Eliminate entries below the pivot
        for r in range(pivot_row + 1, rows):
            if abs(M[pivot_row, col]) > 1e-10:
                factor = M[r, col] / M[pivot_row, col]
                M[r] -= factor * M[pivot_row]
        pivot_row += 1
    return M

print("Function 'row_echelon' is defined and ready to use!")

In [ ]:
# Now let's use it! Build the augmented matrix [A | b]
A = np.array([[ 1,  2, -1,  3],
              [ 2,  1,  3,  7],
              [-1,  3, -2, -2]], dtype=float)

b = np.array([4, 7, 1], dtype=float)

# np.column_stack stacks A and b side by side to form [A | b]
Ab = np.column_stack([A, b])

print("Augmented matrix [A | b]:")
print(Ab)

print("\nRow echelon form:")
print(row_echelon(Ab))



In [ ]:
# ---------------------------------------------------------------
# This function takes matrix A and vector b, row-reduces the
# augmented matrix [A|b], and reports whether the system is
# inconsistent, consistent with a unique solution, or consistent
# with infinitely many solutions (free variables).
# You do NOT need to modify this — just run it to define the function.
# ---------------------------------------------------------------
def check_consistency(A, b):
    """
    Row-reduces the augmented matrix [A | b] and determines whether
    the system Ax = b is inconsistent, uniquely consistent, or has
    infinitely many solutions.
    """
    m, n = A.shape
    Ab = np.column_stack([A, b.astype(float)])
    E  = row_echelon(Ab.copy())

    print("Augmented matrix [A | b]:")
    print(Ab)
    print("\nRow echelon form of [A | b]:")
    print(np.round(E, 6))

    # Check for a bad row: all zeros on A-side but nonzero on b-side
    inconsistent = False
    for row in E:
        if np.allclose(row[:n], 0) and abs(row[n]) > 1e-10:
            inconsistent = True
            break

    if inconsistent:
        print("\nResult: INCONSISTENT")
        print("  A row of the form [0 ... 0 | c] with c ≠ 0 was found.")
        print("  This equation says 0 = c, which is impossible.")
        print("  The system has NO solution.")
        return

    # Count pivot columns in A-part only
    pivot_cols = []
    pivot_row_idx = 0
    for col in range(n):
        for r in range(pivot_row_idx, m):
            if abs(E[r, col]) > 1e-10:
                pivot_cols.append(col)
                pivot_row_idx += 1
                break

    free_cols = [i for i in range(n) if i not in pivot_cols]

    if len(free_cols) == 0:
        print("\nResult: CONSISTENT — unique solution")
        print("  Every column of A has a pivot: no free variables.")
    else:
        print("\nResult: CONSISTENT — infinitely many solutions")
        print(f"  Column(s) {[i+1 for i in free_cols]} of A have no pivot: free variable(s) exist.")

print("Function 'check_consistency' is defined and ready to use!")


### 2.3 Detecting Consistency: Does the System Have a Solution?

You already know the test from class: row-reduce the augmented matrix $[A|\mathbf{b}]$ and inspect the rows.

- If a row looks like $\begin{bmatrix} 0 & 0 & \cdots & 0 & | & c \end{bmatrix}$ with $c \neq 0$, the system is **inconsistent** — that row says $0 = c$, which is impossible.
- If no such row exists, the system is **consistent**. If every column of $A$ has a pivot, the solution is **unique**. If there are columns without pivots, there are **free variables** (infinitely many solutions).

Python can do this automatically using our `row_echelon` function, we just need to inspect the result.

In [ ]:
# --- Example 1: Inconsistent system ---
print("=" * 45)
print("Example 1: Inconsistent system")
print("=" * 45)
A1 = np.array([[1, 2, 3],
               [2, 4, 6],
               [3, 6, 9]], dtype=float)
b1 = np.array([1, 2, 4], dtype=float)
check_consistency(A1, b1)

print()

# --- Example 2: Consistent, unique solution ---
print("=" * 45)
print("Example 2: Consistent -- unique solution")
print("=" * 45)
A2 = np.array([[ 2,  1, -1],
               [-3, -1,  2],
               [-2,  1,  2]], dtype=float)
b2 = np.array([8, -11, -3], dtype=float)
check_consistency(A2, b2)

print()

# --- Example 3: Consistent, infinitely many solutions ---
print("=" * 45)
print("Example 3: Consistent -- free variable")
print("=" * 45)
A3 = np.array([[1, 2, 3],
               [2, 4, 6],
               [3, 6, 9]], dtype=float)
b3 = np.array([1, 2, 3], dtype=float)   # b is now a multiple of the rows -- consistent!
check_consistency(A3, b3)

---

## Part 3: Linear Independence

### 3.1 Testing Linear Independence

Recall from class: the vectors {v1, v2, ..., vk} are **linearly independent** if the only
solution to the equation

$$c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k = \mathbf{0}$$

is $c_1 = c_2 = \cdots = c_k = 0$ (the trivial solution).

To test this with Python, we form the matrix $A$ whose **columns** are the vectors,
then row-reduce $A$ and ask: does $A\mathbf{x} = \mathbf{0}$ have only the trivial solution?

- If every column of the row echelon form has a pivot --> **linearly independent**
- If any column has no pivot (a free variable exists) --> **linearly dependent**

As above, just run the cell below and then we can use the function. 

In [ ]:
def check_independence(vectors, labels=None):
    """
    Tests linear independence by forming the matrix whose columns are
    the given vectors, row-reducing it, and checking whether every
    column has a pivot.  A column with no pivot means a free variable
    exists, so the vectors are linearly dependent.
    """
    if labels is None:
        labels = [f"v{i+1}" for i in range(len(vectors))]

    A = np.column_stack(vectors).astype(float)
    E = row_echelon(A.copy())

    print("Matrix whose columns are the vectors:")
    print(A)
    print("\nRow echelon form:")
    print(np.round(E, 6))

    # Identify which columns have a pivot
    m, n = E.shape
    pivot_cols = []
    pivot_row = 0
    for col in range(n):
        for r in range(pivot_row, m):
            if abs(E[r, col]) > 1e-10:
                pivot_cols.append(col)
                pivot_row += 1
                break

    free_cols = [i for i in range(n) if i not in pivot_cols]

    print()
    if len(free_cols) == 0:
        print("Every column has a pivot -- no free variables.")
        print("The vectors are LINEARLY INDEPENDENT.")
    else:
        print(f"Column(s) {[labels[i] for i in free_cols]} have no pivot -- free variable(s) exist.")
        print("The vectors are LINEARLY DEPENDENT.")
        print("(At least one vector can be written as a combination of the others.)")

print("Function 'check_independence' defined!")

In [ ]:
# --- Example 1: Linearly INDEPENDENT vectors ---
print("=" * 50)
print("Example 1: Three vectors in R^3")
print("=" * 50)
v1 = np.array([1, 0, 0])
v2 = np.array([0, 2, 0])
v3 = np.array([1, 1, 3])
check_independence([v1, v2, v3], ["v1", "v2", "v3"])

print()

# --- Example 2: Linearly DEPENDENT vectors ---
print("=" * 50)
print("Example 2: Three vectors (v3 = 2*v1 + v2)")
print("=" * 50)
w1 = np.array([1, 2, 3])
w2 = np.array([4, 5, 6])
w3 = np.array([6, 9, 12])  # w3 = 2*w1 + w2
check_independence([w1, w2, w3], ["w1", "w2", "w3"])


## Part 4: One-to-One and Onto -- Properties of Linear Maps

For a matrix $A$ of size $m \times n$ (mapping $\mathbb{R}^n \to \mathbb{R}^m$):

| Property | What it means | Test using row echelon form |
|---|---|---|
| **One-to-one** | $A\mathbf{x} = \mathbf{0}$ has only the trivial solution | Every column has a pivot (no free variables) |
| **Onto** | Every $\mathbf{b}$ in $\mathbb{R}^m$ is achievable | Every row has a pivot (no zero rows) |

Both properties depend on counting pivots (the same pivots you find during row reduction).  Run the cell below to define the function and then we will use it.

In [ ]:
def analyze_linear_map(A):
    """
    Determines whether T(x) = Ax is one-to-one, onto, both, or neither
    by row-reducing A and counting pivot rows and pivot columns.
    """
    m, n = A.shape
    E = row_echelon(A.copy().astype(float))

    print(f"Matrix A is {m} x {n}  (maps R^{n} --> R^{m})")
    print("\nRow echelon form of A:")
    print(np.round(E, 6))

    # Count pivots by scanning each column for a leading nonzero entry
    pivot_cols = []
    pivot_row = 0
    for col in range(n):
        for r in range(pivot_row, m):
            if abs(E[r, col]) > 1e-10:
                pivot_cols.append(col)
                pivot_row += 1
                break

    num_pivots = len(pivot_cols)
    free_cols  = [i for i in range(n) if i not in pivot_cols]
    zero_rows  = sum(1 for row in E if np.allclose(row, 0))

    print(f"\nNumber of pivots : {num_pivots}")
    print(f"Number of columns: {n}  (one per variable)")
    print(f"Number of rows   : {m}  (one per equation)")

    print()

    # One-to-one: every column must have a pivot (no free variables)
    if len(free_cols) == 0:
        print("ONE-TO-ONE: yes -- every column has a pivot, so no free variables.")
        print("   Ax = 0 has only the trivial solution.")
    else:
        print(f"ONE-TO-ONE: no -- column(s) {[i+1 for i in free_cols]} have no pivot.")
        print("   Ax = 0 has a nontrivial solution (free variable exists).")

    print()

    # Onto: every row must have a pivot (no zero rows)
    if zero_rows == 0:
        print(f"ONTO: yes -- every row has a pivot, so the columns span R^{m}.")
    else:
        print(f"ONTO: no -- {zero_rows} zero row(s) in echelon form.")
        print(f"   The columns do not span all of R^{m}.")

print("Function 'analyze_linear_map' defined!")

In [ ]:
# --- Example 1: A 3x3 matrix (one-to-one AND onto) ---
print("====== Example 1: 3x3 invertible matrix ======")
A1 = np.array([[1, 0,  2],
               [3, 1,  4],
               [0, 2, -1]], dtype=float)
analyze_linear_map(A1)

print()

# --- Example 2: A 3x2 matrix (more equations than unknowns) ---
print("====== Example 2: 3x2 matrix ======")
A2 = np.array([[1, 2],
               [3, 4],
               [5, 6]], dtype=float)
analyze_linear_map(A2)

print()

# --- Example 3: A 2x3 matrix (more unknowns than equations) ---
print("====== Example 3: 2x3 matrix ======")
A3 = np.array([[1, 0, 2],
               [0, 1, 3]], dtype=float)
analyze_linear_map(A3)

---

## Part 5: Matrix Inverse

### 5.1 Computing and Verifying the Inverse

The inverse of a square matrix $A$ (written $A^{-1}$) satisfies $A A^{-1} = A^{-1} A = I$.

It exists **if and only if** $A$ is invertible. From what you already know, this means:
- Every column of $A$ has a pivot (no free variables)
- Every row of $A$ has a pivot (no zero rows)
- Equivalently: $\det(A) \neq 0$

In other words, $A$ must be square and its row echelon form must have a pivot in
every row and every column.

Python computes the inverse with `np.linalg.inv(A)`.

In [ ]:
A = np.array([[2, 1, 0],
              [1, 3, 1],
              [0, 1, 4]], dtype=float)

# Compute the inverse
A_inv = np.linalg.inv(A)

print("A =\n", A)
print("\nA^-1 =\n", A_inv)

# Verify: A @ A_inv should be the identity matrix I
print("\nA @ A^-1 =\n", np.round(A @ A_inv, 10))  # round to avoid floating-point noise
print("\nYou should see the 3x3 identity matrix!")

# Also verify the other direction: A^-1 @ A should also give I
print("\nA^-1 @ A =\n", np.round(A_inv @ A, 10))
print("\nBoth directions give the identity which that confirms A^-1 is correct.")

### 5.2 Using the Inverse to Solve Systems Efficiently

Once you have $A^{-1}$, solving $A\mathbf{x} = \mathbf{b}$ is just one matrix-vector multiplication:

$$\mathbf{x} = A^{-1}\mathbf{b}$$

This becomes especially powerful when the matrix $A$ stays the same but $\mathbf{b}$ changes.
For example, imagine a network of pipes where the connections (described by $A$) are fixed,
but the flow demands (described by $\mathbf{b}$) change each day. You compute $A^{-1}$ once,
then find each day's solution with a single multiplication instead of row-reducing from scratch every time.

In the example below, we solve three different systems that all share the same matrix $A$.

In [ ]:
# --- Using the inverse to solve multiple systems with the same A ---
# If Ax = b, then x = A^-1 @ b
# This is especially powerful when you need to solve Ax = b for MANY different b vectors

A = np.array([[3, 1],
              [2, 4]], dtype=float)

A_inv = np.linalg.inv(A)

# First right-hand side
b1 = np.array([5, 6], dtype=float)
x1 = A_inv @ b1
print("Solving Ax = b1:")
print("b1 =", b1)
print("x1 = A^-1 @ b1 =", x1)
print("Verify: A @ x1 =", A @ x1, "  (should equal b1)")

print()

# Second right-hand side -- same A, just compute A^-1 @ b2
b2 = np.array([1, 0], dtype=float)
x2 = A_inv @ b2
print("Solving Ax = b2:")
print("b2 =", b2)
print("x2 = A^-1 @ b2 =", x2)
print("Verify: A @ x2 =", A @ x2, "  (should equal b2)")

print()

# Third right-hand side
b3 = np.array([4, -2], dtype=float)
x3 = A_inv @ b3
print("Solving Ax = b3:")
print("b3 =", b3)
print("x3 = A^-1 @ b3 =", x3)
print("Verify: A @ x3 =", A @ x3, "  (should equal b3)")

### 5.2 What Happens with a Singular (Non-Invertible) Matrix?

If $\det(A) = 0$, the matrix has no inverse. Let's see what Python does!

In [ ]:
# A singular matrix: row 2 = 2 x row 1
A_singular = np.array([[1, 2],
                        [2, 4]], dtype=float)

det = np.linalg.det(A_singular)
print(f"det(A) = {det:.6f}")

if abs(det) > 1e-10:
    A_inv = np.linalg.inv(A_singular)
    print("A is invertible.")
    print("A^-1 =\n", A_inv)
else:
    print("det(A) = 0 -- this matrix is NOT invertible.")
    print("Row 2 is a multiple of row 1, so the rows are linearly dependent.")
    print("Always check det(A) is not zero before inverting!")

---

## Part 6: LU Factorization

### 6.1 Computing the LU Decomposition

The **LU factorization** writes $A = LU$ where:  
- $L$ is **lower triangular** (1s on diagonal, entries below)  
- $U$ is **upper triangular** (the row echelon form of $A$)

This is extremely useful for solving $A\mathbf{x} = \mathbf{b}$ efficiently, especially when $A$ stays the same but $\mathbf{b}$ changes for many values.

> **Note:** SciPy's `linalg.lu` returns three matrices: `P, L, U` where $PA = LU$.  
> The matrix $P$ is a **permutation matrix** from row swaps. (For this course we don't explore situations when it does not equal the identity and when $P = I$, we have the standard $A = LU$.

In practice, once we have $A = LU$, solving $A\mathbf{x} = \mathbf{b}$ breaks into two simpler steps:

$$A\mathbf{x} = \mathbf{b} \quad \Longrightarrow \quad L\mathbf{y} = \mathbf{b}, \text{ then } U\mathbf{x} = \mathbf{y}$$

Each of these is easy to solve because $L$ and $U$ are triangular; you just use forward substitution on $L$ (top to bottom) and back substitution on $U$ (bottom to top),
exactly the same process you use at the end of row reduction to find the solution.

For this lab, we will just get the factorization and not solve a system.

In [ ]:
A = np.array([[2, 1,  1],
              [4, 3,  3],
              [8, 7, 9]], dtype=float)

# Compute LU factorization
P, L, U = linalg.lu(A)

print("Original matrix A:")
print(A)

print("\nL (lower triangular):")
print(L)

print("\nU (upper triangular — this is the row echelon form):")
print(U)

print("\nVerification: L @ U should reconstruct A")
print("L @ U =\n", np.round(L @ U, 10))
print("\nA     =\n", A)
print("\nMatch?:", np.allclose(L @ U, A))

---

## Problems

Now it's your turn! Each problem below asks you to **modify the code** from the examples above. The instructions tell you exactly what to change — you don't need to write code from scratch.

**How to work these problems:**  
1. Read the problem carefully.  
2. Change only what is asked (usually the matrix/vector values or a function call).  
3. Run the cell with Shift+Enter and interpret the output.

---

### Problem 1 — Solving a Linear System and Verifying

Consider the system:
$$
\begin{cases}
3x_1 - x_2 + 2x_3 = 10 \\
x_1 + 2x_2 - x_3 = 3 \\
2x_1 + x_2 + 3x_3 = 11
\end{cases}
$$

**(a)** In the cell below, fill in the entries of `A` and `b`, then run the cell to solve the system.  
**(b)** Does the output confirm the solution?  


In [ ]:
# PROBLEM 1 -- YOUR WORK HERE
# Replace the ??? with the correct values from the system above.

A = np.array([[???, ???, ???],   # Row 1 coefficients: 3, -1, 2
              [???, ???, ???],   # Row 2 coefficients
              [???, ???, ???]], dtype=float)  # Row 3 coefficients

b = np.array([???, ???, ???], dtype=float)   # Right-hand sides: 10, 3, 11

# --- Part (a): Solve the system ---
x = np.linalg.solve(A, b)
print("Solution:")
print(f"  x1 = {x[0]:.4f}")
print(f"  x2 = {x[1]:.4f}")
print(f"  x3 = {x[2]:.4f}")

# --- Part (b): Verify ---
print("\nVerification:")
print("  A @ x =", np.round(A @ x, 6))
print("  b     =", b)


*Double-click here and type your response.*


### Problem 2 -- Linear Independence

Consider the following two sets of vectors. For each set, use the
`check_independence` function to determine if they are linearly independent or dependent.

**(a)** $\mathbf{v}_1 = \begin{bmatrix}1\\2\\3\end{bmatrix}$,
$\mathbf{v}_2 = \begin{bmatrix}0\\1\\4\end{bmatrix}$,
$\mathbf{v}_3 = \begin{bmatrix}5\\6\\7\end{bmatrix}$

**(b)** $\mathbf{w}_1 = \begin{bmatrix}1\\0\\0\end{bmatrix}$,
$\mathbf{w}_2 = \begin{bmatrix}0\\1\\0\end{bmatrix}$,
$\mathbf{w}_3 = \begin{bmatrix}0\\0\\1\end{bmatrix}$
(Hint: do you recognize these vectors? Can you predict the answer before running the code?)



In [ ]:
# PROBLEM 2 — YOUR WORK HERE
# Fill in the vector components for parts (a) and (b).

# --- Part (a) ---
print("Part (a):")
v1 = np.array([???, ???, ???])   # Fill in v1
v2 = np.array([???, ???, ???])   # Fill in v2
v3 = np.array([???, ???, ???])   # Fill in v3
check_independence([v1, v2, v3], ["v1", "v2", "v3"])

print()

# --- Part (b) ---
print("Part (b):")
w1 = np.array([???, ???, ???])   # Fill in w1
w2 = np.array([???, ???, ???])   # Fill in w2
w3 = np.array([???, ???, ???])   # Fill in w3
check_independence([w1, w2, w3], ["w1", "w2", "w3"])

*Double-click here and type your response.*

### Problem 3 -- One-to-One and Onto

For each matrix below, use `analyze_linear_map` to determine if the map is
one-to-one, onto, both, or neither. Then explain **why** in terms of pivots, specifically, which columns or rows are missing a pivot.

**(a)** $A = \begin{bmatrix}1 & 0 & 2 & 1\\ 0 & 1 & -1 & 3\\ 2 & 1 & 3 & 5\end{bmatrix}$ (a $3 \times 4$ matrix)

**(b)** $A = \begin{bmatrix}1 & 2\\ 3 & 4\\ 5 & 6\\ 7 & 8\end{bmatrix}$ (a $4 \times 2$ matrix)



In [ ]:
# PROBLEM 3 -- YOUR WORK HERE
# Fill in the matrix entries.

# --- Part (a): 3x4 matrix ---
print("Part (a):")
A_a = np.array([[???, ???, ???, ???],   # Row 1
                [???, ???, ???, ???],   # Row 2
                [???, ???, ???, ???]], dtype=float)  # Row 3
analyze_linear_map(A_a)

print()

# --- Part (b): 4x2 matrix ---
print("Part (b):")
A_b = np.array([[???, ???],   # Row 1
                [???, ???],   # Row 2
                [???, ???],   # Row 3
                [???, ???]], dtype=float)  # Row 4
analyze_linear_map(A_b)

*Double-click here and type your response.*

**Explanation for (a):** ______  
**Explanation for (b):** ______

---

### Problem 4 — Matrix Inverse and Solving Systems

Let $A = \begin{bmatrix}5 & 2 & 0\\ 2 & 1 & 0\\ 0 & 0 & 3\end{bmatrix}$.

**(a)** Compute $A^{-1}$ and verify that $A A^{-1} = I$.  
**(b)** Use $A^{-1}$ to solve $A\mathbf{x} = \mathbf{b}$ for the two right-hand sides:
$$\mathbf{b}_1 = \begin{bmatrix}1\\0\\3\end{bmatrix}, \qquad \mathbf{b}_2 = \begin{bmatrix}4\\2\\9\end{bmatrix}$$
**(c)** Verify each solution by computing $A\mathbf{x}$.

In [ ]:
# PROBLEM 4 -- YOUR WORK HERE

# --- Part (a): Define A and compute its inverse ---
A = np.array([[???, ???, ???],
              [???, ???, ???],
              [???, ???, ???]], dtype=float)

A_inv = np.linalg.inv(A)
print("A^-1 =\n", A_inv)
print("\nA @ A^-1 =\n", np.round(A @ A_inv, 10))  # should be the identity matrix
print("\nA^-1 @ A =\n", np.round(A_inv @ A, 10))  # should also be the identity matrix

# --- Part (b): Solve for b1 and b2 ---
b1 = np.array([???, ???, ???], dtype=float)
b2 = np.array([???, ???, ???], dtype=float)

x1 = A_inv @ b1
x2 = A_inv @ b2

print("\nx1 (solution for b1) =", x1)
print("x2 (solution for b2) =", x2)

# --- Part (c): Verify ---
print("\nVerification:")
print("A @ x1 =", A @ x1, "  (should equal b1 =", b1, ")")
print("A @ x2 =", A @ x2, "  (should equal b2 =", b2, ")")

*Double-click here and type your response.*

---

### Problem 5 -- LU Factorization

Let $A = \begin{bmatrix}4 & 3 & 0\\ 8 & 9 & 2\\ 0 & 6 & 5\end{bmatrix}$.

**(a)** Compute the LU factorization of $A$ and print $L$ and $U$.

**(b)** Verify by computing $L \cdot U$ and comparing it to $A$. You may notice
that $L \cdot U$ does not exactly equal $A$ but instead equals $P \cdot A$, where
$P$ is a matrix that simply reorders the rows of $A$. This row reordering happens
automatically inside Python for numerical stability. 

In [ ]:
# PROBLEM 5 -- YOUR WORK HERE

# --- Part (a): Define A and compute LU factorization ---
A = np.array([[???, ???, ???],
              [???, ???, ???],
              [???, ???, ???]], dtype=float)

P, L, U = linalg.lu(A)

print("L (lower triangular):")
print(L)
print("\nU (upper triangular):")
print(U)

# --- Part (b): Verify ---
print("\nL @ U =\n", np.round(L @ U, 10))
print("\nP @ A =\n", np.round(P @ A, 10))
print("\nDo L @ U = P @ A?", np.allclose(L @ U, P @ A))

*Double-click here and type your response.*

## Congratulations!

You have worked through many of the major topics from your linear algebra course using Python:

| Topic | Python Tool Used |
|-------|------------------|
| Vectors and matrix equations | `np.array`, `@` operator |
| Solving linear systems | `np.linalg.solve` |
| Row echelon form | Custom `row_echelon` function |
| Consistency (bad row test) | Custom `check_consistency` function |
| Linear independence | Custom `check_independence` function |
| One-to-one and onto | Custom `analyze_linear_map` function |
| Matrix inverse | `np.linalg.inv` |
| LU factorization | `scipy.linalg.lu`  |

**Key takeaway:** Computers don't replace the mathematical reasoning you've developed, but they let you apply it at scale. Understanding *why* these algorithms work (row
operations, pivot positions, linear independence) is what lets you interpret the output and catch errors. The custom functions in this notebook are all built on
the same row reduction ideas you have been using by hand all semester.

**You can reaccess your notebook through Jupyter Hub:** You are welcome to copy and modify this notebook if you ever want to check your homework or do calculations with Python.

**You will not be tested on coding on exams in this class.**  This is just a cool and practical tool that I wanted you to see.

## Submitting Your Work

When you have finished working through the notebook, follow these steps to submit on Canvas:

1. Make sure you have run every cell in order from top to bottom and that all
   output is visible below each code cell.
2. Go to **File** in the top menu.
3. Select **Save and Export Notebook As** and then choose **PDF**.  This will save the PDF locally on your computer.
4. Upload this PDF to the assignment on Canvas.

**Before submitting, double check:**
- All code cells have been run and show output.
- Your written answers are filled in for each problem.  (For example, below Problem 3, double click and type your response.)